<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_6/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_6_3_%D0%92%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B2_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.3. Введение в LangChain

## Введение: от самодельного RAG к промышленному фреймворку

Поздравляю! Мы прошли огромный путь. В Лекции 6.1 мы создали игрушечного RAG-агента с нуля — вручную отправляли HTTP-запросы к Ollama, писали свой семантический поиск через `cosine_similarity` и маршрутизировали по ключевым словам. В Лекции 6.2 мы превратили его в полноценную систему: загрузка документов, умный чанкинг, векторная база Chroma, интеллектуальная маршрутизация через LLM, память и логирование. Всё это мы сделали **на чистом Python, без фреймворков**, чтобы понять, как всё работает «под капотом».

Теперь настало время **сделать шаг вперёд**. Мы освоим **LangChain** — мощный фреймворк для разработки приложений на основе больших языковых моделей. Он не заменит наше понимание, а **возьмёт на себя рутину**: управление промптами, цепочками вызовов, памятью, интеграцию с векторными базами и многое другое. LangChain позволяет сократить объём кода в 3–5 раз, делая его более читаемым, гибким и легко расширяемым.

**Что мы изучим в этой лекции:**
- **Компоненты LangChain** — LLM, промпты, цепочки, ретриверы, память.
- **LCEL (LangChain Expression Language)** — новый синтаксис для построения пайплайнов.
- **Вызов LLM через ChatOllama** — вместо `requests.post` одна строка кода.
- **Промпты и парсеры** — структурированное управление шаблонами и форматированием ответов.
- **Цепочки** — объединение промпта и LLM в единый конвейер.
- **Ретриверы и векторные хранилища** — интеграция с Chroma через LangChain.
- **Память** — управление историей диалога через встроенные классы.
- **Сборка полноценного RAG-агента** на LangChain.

Мы перепишем ту же систему, что строили в Лекции 6.2, но теперь с использованием абстракций LangChain. Вы увидите, как сокращается код и возрастает его выразительность.

---

## Тема 1. Что такое LangChain и зачем он нужен

LangChain — это фреймворк с открытым исходным кодом, созданный для упрощения разработки приложений на основе LLM. Он предоставляет стандартизированные интерфейсы для работы с моделями, данными и логикой приложений.

### 1.1. Ключевые компоненты LangChain

| Компонент | Назначение | В нашей системе из Лекции 6.2 |
|-----------|------------|-------------------------------|
| **LLM** | Интерфейс для языковых моделей (Ollama, OpenAI, Anthropic) | Мы делали `requests.post` вручную |
| **Промпт** | Управление шаблонами и форматированием | Мы вручную собирали строки с f-строками |
| **Цепочки (Chains)** | Объединение нескольких шагов в один пайплайн | Мы писали отдельные функции и вызывали их последовательно |
| **Ретриверы** | Поиск релевантных документов (интерфейс для векторных БД) | Мы сами писали `search_chunks` |
| **Память (Memory)** | Хранение истории диалога | Мы реализовали `ConversationMemory` вручную |
| **Парсеры (Parsers)** | Извлечение структурированных данных из ответов LLM | Мы парсили JSON вручную с `json.loads` |

Каждый компонент предоставляет **стандартизированный интерфейс**. Это значит, что мы можем легко заменить одну модель на другую, одну векторную БД на другую, не переписывая остальной код.

### 1.2. LCEL — LangChain Expression Language

LCEL — это синтаксический сахар для построения цепочек. Вместо того чтобы писать:

```python
prompt = PromptTemplate(...)
chain = prompt | llm | parser
result = chain.invoke({"input": "вопрос"})
```

Мы используем оператор `|` для объединения компонентов. Это напоминает конвейеры в Unix (`|`) и делает код **декларативным** — мы описываем **что** делаем, а не **как**.

**Пример простейшей цепочки:**

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:3b")

prompt = ChatPromptTemplate.from_template("Ответь на вопрос: {question}")

chain = prompt | llm

response = chain.invoke({"question": "Что такое LangChain?"})
print(response.content)
```

Этот код делает ровно то же, что и 10 строк с `requests.post`, но лаконичнее и стандартизированнее.

### 1.3. Установка пакетов

Для работы с LangChain в нашем проекте установим необходимые пакеты:

```bash
pip install langchain langchain-community langchain-chroma langchain-ollama
```

**Что мы установили:**

| Пакет | Назначение |
|-------|------------|
| `langchain` | Ядро фреймворка (базовые интерфейсы) |
| `langchain-community` | Интеграции с различными сервисами (Chroma, HuggingFace, и др.) |
| `langchain-chroma` | Специализированный пакет для работы с Chroma через LangChain |
| `langchain-ollama` | Интеграция с Ollama через стандартный интерфейс LangChain |

Теперь проверим, что всё установилось корректно:

```bash
python -c "import langchain; import langchain_ollama; print('✅ LangChain готов к работе')"
```

### 1.4. Обзорная схема: как всё будет связано

Вот как будет выглядеть наша RAG-система на LangChain:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                          ПОЛЬЗОВАТЕЛЬ                                      │
│                                │                                             │
│                           Вопрос: "Что такое ProjectFlow?"                  │
└────────────────────────────────┬────────────────────────────────────────────┘
                                 ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│                         CHAIN (ЦЕПОЧКА)                                   │
│                                                                             │
│   ┌───────────────────────────────────────────────────────────────────┐    │
│   │  1. ПРОМПТ С ИСТОРИЕЙ                                            │    │
│   │  - Шаблон: "ИСТОРИЯ: {history}\nВОПРОС: {question}\nОТВЕТ:"     │    │
│   └───────────────────────────────────────────────────────────────────┘    │
│                                   │                                         │
│                                   ▼                                         │
│   ┌───────────────────────────────────────────────────────────────────┐    │
│   │  2. РЕТРИВЕР (VectorStoreRetriever)                              │    │
│   │  - Ищет в Chroma топ‑3 чанка по вопросу                         │    │
│   │  - Возвращает: [документ1, документ2, документ3]                │    │
│   └───────────────────────────────────────────────────────────────────┘    │
│                                   │                                         │
│                                   ▼                                         │
│   ┌───────────────────────────────────────────────────────────────────┐    │
│   │  3. ПРОМПТ С КОНТЕКСТОМ (RAG)                                    │    │
│   │  - Шаблон: "КОНТЕКСТ: {context}\nВОПРОС: {question}"             │    │
│   │  - Подставляем найденные чанки                                    │    │
│   └───────────────────────────────────────────────────────────────────┘    │
│                                   │                                         │
│                                   ▼                                         │
│   ┌───────────────────────────────────────────────────────────────────┐    │
│   │  4. LLM (ChatOllama)                                             │    │
│   │  - Отправляет запрос в qwen2.5:3b                                │    │
│   └───────────────────────────────────────────────────────────────────┘    │
│                                   │                                         │
│                                   ▼                                         │
│   ┌───────────────────────────────────────────────────────────────────┐    │
│   │  5. ПАРСЕР (StrOutputParser)                                     │    │
│   │  - Извлекает response.content в виде строки                      │    │
│   └───────────────────────────────────────────────────────────────────┘    │
└─────────────────────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│                          ПОЛЬЗОВАТЕЛЬ                                      │
│                                                                             │
│                    Ответ: "ProjectFlow — это облачная платформа..."        │
└─────────────────────────────────────────────────────────────────────────────┘
```

**Вся цепочка на LangChain будет выглядеть примерно так:**

```python
chain = (
    {"context": retriever, "question": lambda x: x["question"]}
    | rag_prompt
    | llm
    | StrOutputParser()
)
```

И это **вся логика RAG**. Остальное — загрузка документов и настройка векторной базы, но это тоже делается в несколько строк.

Теперь, когда мы понимаем концепцию, давайте начнём с самого простого — вызова LLM через LangChain.

---




## Тема 2. Вызов LLM через ChatOllama (продолжение — подробно о параметрах)

В предыдущем разделе мы кратко перечислили параметры `ChatOllama`. Теперь давайте разберём каждый из них **максимально подробно** — как они работают, какие значения принимать и как влияют на результат.

### 2.6. Полный список параметров ChatOllama

```python
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:3b",           # Имя модели
    temperature=0.0,               # 0 = детерминированно, 1 = креативно
    num_predict=1024,              # Максимум токенов в ответе
    top_k=40,                      # Ограничение на выбор топ-K токенов
    top_p=0.9,                     # Ядерная выборка (nucleus sampling)
    repeat_penalty=1.1,            # Штраф за повторения
    stop=["\n", "Вопрос:"],        # Стоп-слова
    timeout=60,                    # Таймаут в секундах
    num_ctx=4096,                  # Размер контекстного окна (токены)
    seed=None,                     # Детерминированность (фиксированный seed)
    tfs_z=1.0,                     # Tail-free sampling (Z-значение)
    mirostat_mode=0,               # Режим Mirostat (0=выкл, 1=вкл, 2=экспериментальный)
    mirostat_tau=5.0,              # Температура Mirostat
    mirostat_eta=0.1,              # Скорость обучения Mirostat
)
```

Теперь разберём каждый параметр.

---

### 2.6.1. `model` — выбор модели

**Что это:** строка с именем модели, загруженной в Ollama.

**Примеры:**
- `"qwen2.5:3b"`
- `"llama3.1:8b"`
- `"mistral:7b"`

**Как использовать:** перед инициализацией убедитесь, что модель скачана:

```bash
ollama pull qwen2.5:3b
```

Если модель не найдена, вы получите ошибку соединения.

**Почему важно:** разные модели имеют разный размер, качество и скорость. Для экспериментов подойдёт `qwen2.5:3b` (2 ГБ), для серьёзных задач — `llama3.1:8b` (4.7 ГБ).

---

### 2.6.2. `temperature` — контроль случайности

**Что это:** число от 0 до 2 (чаще 0–1). Влияет на креативность модели.

| Значение | Эффект |
|----------|--------|
| `0.0` | **Детерминированно** — каждый раз один и тот же ответ (при прочих равных). Идеально для точных фактов, кода, математики. |
| `0.3–0.5` | **Сбалансированно** — небольшая вариативность. Хорошо для чатов, когда нужна естественность, но без фантазий. |
| `0.7–0.9` | **Креативно** — модель генерирует разные варианты. Для творческих задач, генерации идей, сторителлинга. |
| `1.0+` | **Очень креативно** — может уходить в сторону, иногда галлюцинировать. |

**Как это работает технически:** LLM на каждом шаге предсказывает вероятности для всех токенов. `temperature` масштабирует эти вероятности — чем выше значение, тем «площе» распределение, и модель с большей вероятностью выбирает не самый очевидный токен. При `temperature=0` выбирается токен с максимальной вероятностью (жадный поиск).

**Пример:**

```python
# Детерминированный ответ (всегда один)
llm_det = ChatOllama(model="qwen2.5:3b", temperature=0.0)
for _ in range(3):
    print(llm_det.invoke("Скажи число 2+2").content)  # всегда "4"

# Креативный ответ (может меняться)
llm_creative = ChatOllama(model="qwen2.5:3b", temperature=0.9)
for _ in range(3):
    print(llm_creative.invoke("Придумай имя для кота").content)
    # Вывод: "Барсик", "Мурзик", "Снежок" — разные варианты
```

---

### 2.6.3. `num_predict` — максимальная длина ответа

**Что это:** максимальное количество токенов в ответе (не включая промпт).

| Значение | Применение |
|----------|------------|
| `128` | Короткие ответы (да/нет, одно слово, краткая справка) |
| `512` | Стандартный ответ средней длины (параграф) |
| `1024` | Развёрнутый ответ (несколько абзацев) |
| `2048+` | Длинные тексты (статьи, обзоры, код) |

**Важно:** не путать с размером контекстного окна (`num_ctx`). `num_predict` ограничивает только **генерируемый ответ**, а `num_ctx` — объём входных данных, которые модель может «увидеть» (включая промпт и историю).

**Пример:** если промпт занимает 3000 токенов, а `num_ctx=4096`, то модель может сгенерировать ещё 1096 токенов. Но если `num_predict=512`, то ответ будет обрезан после 512 токенов, даже если модель могла бы продолжить.

---

### 2.6.4. `top_k` — фильтрация по количеству

**Что это:** число (обычно 10–100). Модель рассматривает только `top_k` наиболее вероятных токенов на каждом шаге.

| Значение | Эффект |
|----------|--------|
| `1` | Жадный поиск (всегда самый вероятный токен) — аналог `temperature=0` |
| `10–20` | Умеренное ограничение — модель выбирает из небольшого пула лучших токенов |
| `40–100` | Широкий выбор — больше разнообразия |
| `0` (или очень большое) | Отключает фильтрацию — рассматриваются все токены (медленнее) |

**Как работает:** на каждом шаге модель вычисляет вероятности для всех токенов (десятки тысяч). `top_k` оставляет только K самых вероятных, остальные обнуляет. Это ускоряет генерацию и убирает маловероятный «мусор».

**Пример:** при `top_k=10` модель никогда не выберет редкое слово, даже если оно контекстуально подходит, если оно не входит в топ‑10. При `top_k=40` такой токен имеет шанс.

---

### 2.6.5. `top_p` — ядерная выборка (nucleus sampling)

**Что это:** число от 0 до 1 (обычно 0.8–0.95). Модель выбирает минимальный набор токенов, чья суммарная вероятность превышает `top_p`.

| Значение | Эффект |
|----------|--------|
| `0.1` | Очень строго — только самые вероятные токены (почти детерминированно) |
| `0.5` | Умеренно — половина вероятностной массы |
| `0.9` | Широкий выбор — 90% массы (обычно это десятки–сотни токенов) |
| `1.0` | Без ограничений (эквивалент `top_p=1`) |

**Разница между `top_k` и `top_p`:**
- `top_k` — фильтрует по **количеству** токенов (фиксированное число).
- `top_p` — фильтрует по **кумулятивной вероятности** (динамическое число).

Обычно используют **либо `top_k`, либо `top_p`**, но можно комбинировать: сначала `top_k` сужает список, затем `top_p` отсекает по вероятности.

**Пример:** если вероятности токенов [0.5, 0.3, 0.15, 0.05], то при `top_p=0.9` будут выбраны первые три (сумма 0.95), четвёртый отброшен.

---

### 2.6.6. `repeat_penalty` — штраф за повторения

**Что это:** число от 1.0 до 2.0 (обычно 1.05–1.2). Уменьшает вероятность повторения уже использованных токенов.

| Значение | Эффект |
|----------|--------|
| `1.0` | Штраф выключен — модель может повторяться |
| `1.05–1.1` | Лёгкий штраф — немного уменьшает повторения |
| `1.2–1.5` | Сильный штраф — активно избегает повторений |
| `2.0+` | Почти запрещает повторения (может сломать логику) |

**Как работает:** модель умножает вероятности токенов, которые уже встречались в контексте, на `1/repeat_penalty`. Чем выше штраф, тем меньше шанс выбрать повторяющийся токен.

**Пример:**
- `repeat_penalty=1.0`: модель может написать «Я думаю, что я думаю...»
- `repeat_penalty=1.2`: такая фраза маловероятна.

---

### 2.6.7. `stop` — стоп-слова

**Что это:** список строк, при обнаружении которых генерация останавливается.

**Пример:** `stop=["\n", "Вопрос:"]` — ответ будет оборван, как только встретится символ новой строки или слово «Вопрос:».

**Применение:**
- Чтобы ограничить ответ одним предложением (`stop=["."]`).
- Чтобы запретить модели задавать встречные вопросы (`stop=["?"]`).
- Чтобы разделять несколько ответов в одном запросе.

---

### 2.6.8. `timeout` — таймаут запроса

**Что это:** максимальное время ожидания ответа от Ollama в секундах.

**Значение:** если модель не отвечает за `timeout` секунд, выбрасывается исключение `TimeoutError`.

**Рекомендация:** для коротких промптов — 30 сек, для длинных генераций (например, статей) — 120+ сек.

---

### 2.6.9. `num_ctx` — размер контекстного окна

**Что это:** количество токенов, которое модель может «видеть» (вход + выход вместе). Это **не** то же самое, что `num_predict`.

| Значение | Применение |
|----------|------------|
| `2048` | Минимальное (многие старые модели) |
| `4096` | Стандарт для многих моделей (qwen2.5:3b поддерживает 8192) |
| `8192` | Расширенное окно (для длинных документов) |
| `32768` | Для моделей с большим контекстом (Llama 3.1) |

**Важно:** если суммарный объём (промпт + история + ответ) превышает `num_ctx`, модель обрежет начало промпта. Поэтому для RAG-систем с большими документами нужно увеличивать `num_ctx`.

---

### 2.6.10. `seed` — детерминизм

**Что это:** целое число (или `None`). Задаёт начальное состояние генератора случайных чисел.

| Значение | Эффект |
|----------|--------|
| `None` | Случайный seed — ответы будут различаться при каждом запуске (даже при `temperature=0`) |
| `42` | Фиксированный seed — ответы будут идентичны при одинаковых параметрах |

**Применение:** для воспроизводимости экспериментов, отладки, тестирования.

---

### 2.6.11. `tfs_z` — Tail-free sampling

**Что это:** число (обычно 1.0). Альтернативный метод фильтрации токенов.

- `1.0` — выключено.
- `<1.0` — более жёсткая фильтрация (отбрасывает «хвост» распределения).

Редко используется, оставьте по умолчанию.

---

### 2.6.12. `mirostat_mode`, `mirostat_tau`, `mirostat_eta` — алгоритм Mirostat

**Что это:** экспериментальный алгоритм управления случайностью, предложенный исследователями Microsoft. Позволяет автоматически подстраивать температуру во время генерации, чтобы достичь заданной «энтропии» (разнообразия).

- `mirostat_mode=0` — выключено.
- `mirostat_mode=1` — включён (рекомендуется для творческих задач).
- `mirostat_tau` — целевая энтропия (обычно 5.0). Чем выше, тем разнообразнее текст.
- `mirostat_eta` — скорость адаптации (0.1 — стандарт).

**Применение:** для генерации художественных текстов, где нужно балансировать между разнообразием и связностью.

---

### 2.7. Практические рекомендации по выбору параметров

| Сценарий | Параметры |
|----------|-----------|
| **Точные факты (RAG, QA)** | `temperature=0.0`, `top_k=1` или `top_p=0.1`, `repeat_penalty=1.0` |
| **Код (генерация, объяснение)** | `temperature=0.2`, `top_p=0.5`, `repeat_penalty=1.05` |
| **Чат (естественный диалог)** | `temperature=0.7`, `top_p=0.9`, `repeat_penalty=1.1` |
| **Креативное письмо** | `temperature=0.9`, `top_p=0.95`, `mirostat_mode=1`, `mirostat_tau=5.0` |
| **Сжатие/пересказ** | `temperature=0.3`, `num_predict=256`, `stop=["."]` |

---

### 2.8. Полный код с демонстрацией параметров

Создадим файл `test_llm_params.py`:

```python
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# Набор конфигураций
configs = [
    {"name": "Точный (детерминированный)", "temp": 0.0, "top_k": 1},
    {"name": "Сбалансированный", "temp": 0.5, "top_p": 0.8},
    {"name": "Креативный", "temp": 0.9, "top_p": 0.95},
]

question = "Придумай название для стартапа, который делает ИИ-помощника для программистов."

for cfg in configs:
    llm = ChatOllama(
        model="qwen2.5:3b",
        temperature=cfg["temp"],
        top_k=cfg.get("top_k", 40),
        top_p=cfg.get("top_p", 0.9),
        repeat_penalty=1.1,
        num_predict=128,
        seed=42  # Фиксируем seed для воспроизводимости (кроме температуры)
    )
    
    response = llm.invoke([HumanMessage(content=question)])
    print(f"\n{cfg['name']} (temp={cfg['temp']}):")
    print(response.content)
    print("-" * 60)
```

**Вывод:**
```
Точный (детерминированный) (temp=0.0):
CodeMate

------------------------------------------------------------
Сбалансированный (temp=0.5):
DevPal AI

------------------------------------------------------------
Креативный (temp=0.9):
СodeCraft AI — твой интеллектуальный партнёр в мире программирования!

------------------------------------------------------------
```

Мы видим, как меняется ответ в зависимости от параметров.

---

## Краткий итог по параметрам

| Параметр | Влияние на генерацию |
|----------|----------------------|
| `temperature` | Креативность (чем выше, тем разнообразнее) |
| `num_predict` | Длина ответа |
| `top_k`/`top_p` | Фильтрация вероятностей (чем строже, тем детерминированнее) |
| `repeat_penalty` | Борьба с повторениями |
| `stop` | Принудительная остановка генерации |
| `num_ctx` | Объём контекста (вход + выход) |
| `seed` | Воспроизводимость |

Теперь, когда мы вооружены знанием каждого параметра, можем тонко настраивать модель под любую задачу. В следующей теме мы перейдём к **промптам LangChain** — ещё одному мощному инструменту, который делает наш код компактным и выразительным.



## Тема 3. Промпты и парсеры: от ручного форматирования к структурированным шаблонам

В Лекциях 6.1 и 6.2 мы формировали промпты вручную, используя f-строки и многострочные литералы:

```python
prompt = f"""
Ты — строгий помощник. Отвечай на основе контекста.

Контекст: {context}
Вопрос: {question}
Ответ:
"""
```

Этот подход, хотя и работает, обладает рядом фундаментальных недостатков, которые становятся критическими по мере роста сложности системы:

1. **Смешение логики и представления.** Шаблон промпта размазан по коду, его трудно редактировать, тестировать и поддерживать. Изменение структуры промпта требует правки кода, что нарушает принцип разделения ответственности.

2. **Отсутствие структурной типизации.** Мы не различаем системные инструкции, историю диалога и текущий вопрос — всё смешивается в единую строку. Это особенно проблематично для чат-моделей, которые ожидают структурированный диалог.

3. **Ручной парсинг ответов.** Для маршрутизации мы вручную парсили JSON с помощью `json.loads()` и обрабатывали ошибки через `try/except`. Это порождает шаблонный код, который сложно поддерживать.

4. **Нет валидации.** Мы не проверяли, что модель вернула корректный JSON с нужными полями. Любая ошибка в ответе модели приводила либо к падению, либо к неявному fallback-значению.

5. **Сложность переиспользования.** Один и тот же промпт невозможно легко применить к разным сценариям без копирования кода.

**LangChain предлагает элегантное решение всех этих проблем через три ключевых механизма:**

1. **Шаблоны промптов** (`ChatPromptTemplate`) — отделяют структуру промпта от логики приложения, позволяют задавать разные роли (система, пользователь, ассистент) и подставлять переменные.

2. **Парсеры** (`StrOutputParser`, `PydanticOutputParser`) — автоматически извлекают и валидируют ответы модели, превращая их в типизированные объекты Python.

3. **Интеграция с LCEL** — позволяет собирать промпты, парсеры и модели в единые цепочки, делая код декларативным и легко тестируемым.

В этой теме мы детально разберём каждый из этих механизмов и увидим, как они трансформируют наш подход к разработке RAG-систем.

---

### 3.1. ChatPromptTemplate: создание структурированных шаблонов

`ChatPromptTemplate` — это класс для создания промптов из нескольких сообщений с возможностью подстановки переменных. В отличие от обычных строк, он поддерживает разделение ролей и структурную типизацию.

#### Базовое использование

Самый простой способ — использование фабричного метода `from_template()`:

```python
from langchain_core.prompts import ChatPromptTemplate

# Шаблон с одной переменной
prompt = ChatPromptTemplate.from_template("Ответь на вопрос: {question}")

# Шаблон с несколькими переменными
prompt = ChatPromptTemplate.from_template(
    "Контекст: {context}\nВопрос: {question}\nОтвет:"
)

# Подстановка значений
formatted = prompt.format(question="Что такое LangChain?")
print(formatted)
```

**Вывод:**
```
Ответь на вопрос: Что такое LangChain?
```

#### Создание шаблонов с несколькими сообщениями

Наиболее мощный вариант — создание шаблона из нескольких сообщений с явным указанием ролей. Это соответствует формату, который ожидают современные чат-модели:

```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — эксперт по {topic}. Отвечай кратко и чётко."),
    ("human", "Объясни: {concept}"),
])

# Подстановка
formatted = prompt.format(topic="Python", concept="декоратор")
print(formatted)
```

**Вывод:**
```
System: Ты — эксперта по Python. Отвечай кратко и чётко.
Human: Объясни: декоратор
```

#### Преимущества перед f-строками

| Аспект | f-строки (ручные) | ChatPromptTemplate |
|--------|-------------------|-------------------|
| **Разделение ролей** | Отсутствует | Чёткое (system, human, assistant) |
| **Переиспользование** | Копирование кода | Один шаблон — много вызовов |
| **Поддержка** | Трудно изменять | Легко редактировать |
| **Тестируемость** | Низкая | Высокая (шаблоны изолированы) |
| **Интеграция с LCEL** | Нет | Полная поддержка |

#### Форматирование с явными сообщениями

Иногда удобнее работать с объектами сообщений напрямую. LangChain предоставляет классы `SystemMessage`, `HumanMessage`, `AIMessage`:

```python
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Создаём список сообщений вручную
messages = [
    SystemMessage(content="Ты — помощник по программированию."),
    HumanMessage(content="Что такое класс в Python?"),
]

# Передаём в LLM
response = llm.invoke(messages)
print(response.content)
```

Этот подход особенно полезен, когда мы динамически строим историю диалога, добавляя сообщения ассистента между сообщениями пользователя.

---

### 3.2. Использование SystemMessage, HumanMessage, AIMessage для явного указания ролей

В LangChain каждое сообщение имеет явную роль, что критически важно для корректной работы чат-моделей. Рассмотрим три основных типа:

**1. SystemMessage (Системное сообщение)**

Задаёт контекст, правила и ограничения для модели. Обычно находится в начале диалога и определяет поведение модели на протяжении всей сессии.

```python
from langchain_core.messages import SystemMessage

system_msg = SystemMessage(content="Ты — эксперт по Python. Отвечай кратко и только по теме.")
```

**2. HumanMessage (Сообщение пользователя)**

Представляет запрос или вопрос пользователя. Модель должна ответить на это сообщение.

```python
from langchain_core.messages import HumanMessage

human_msg = HumanMessage(content="Что такое декоратор в Python?")
```

**3. AIMessage (Сообщение ассистента)**

Представляет ответ модели. Используется в истории диалога для сохранения контекста.

```python
from langchain_core.messages import AIMessage

ai_msg = AIMessage(content="Декоратор — это функция, которая принимает другую функцию...")
```

**Комбинирование с ChatPromptTemplate:**

```python
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="Ты — эксперт по {topic}."),
    HumanMessage(content="{question}"),
])

# Подстановка
formatted = prompt.format_messages(topic="Python", question="Что такое декоратор?")
# formatted — список сообщений, готовых для передачи в LLM
```

**Почему это важно:** Многие модели (особенно из семейства Llama, Qwen) чувствительны к структуре диалога. Явное разделение ролей помогает модели лучше понимать, что от неё требуется.

---

### 3.3. PydanticOutputParser: автоматический парсинг структурированных ответов

Одна из самых мощных возможностей LangChain — автоматический парсинг ответов в заданную структуру с помощью Pydantic. Это полностью устраняет необходимость вручную парсить JSON и обрабатывать ошибки.

#### Что такое Pydantic?

Pydantic — это библиотека для валидации данных через Python-классы с аннотациями типов. Она автоматически проверяет, что данные соответствуют заданной структуре, и преобразует их в объекты Python.

#### Создание модели данных

Определим структуру для маршрутизации запросов:

```python
from pydantic import BaseModel, Field
from typing import Literal

class RouterOutput(BaseModel):
    """Структура ответа маршрутизатора."""
    action: Literal["search", "answer"] = Field(
        description="Действие: search — искать в документах, answer — ответить из знаний"
    )
    confidence: float = Field(
        description="Уверенность в решении (0.0 — 1.0)",
        ge=0.0,  # минимальное значение
        le=1.0   # максимальное значение
    )
```

**Разбор полей:**
- `action` — строка, которая может принимать только два значения: `"search"` или `"answer"` (используем `Literal` для ограничения).
- `confidence` — число с плавающей точкой от 0 до 1 (используем `ge` и `le` для ограничения диапазона).
- `Field(description=...)` — добавляет описание, которое автоматически попадает в инструкцию для модели.

#### Создание парсера

```python
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=RouterOutput)
```

#### Получение инструкций для модели

Парсер автоматически генерирует инструкции по форматированию ответа:

```python
format_instructions = parser.get_format_instructions()
print(format_instructions)
```

**Вывод (упрощённый):**
```
The output should be formatted as a JSON instance that conforms to the JSON schema below.

Here is the output schema:
{"properties": {"action": {"description": "Action: search — искать в документах, answer — ответить из знаний", "enum": ["search", "answer"], "title": "Action", "type": "string"}, "confidence": {"description": "Уверенность в решении (0.0 — 1.0)", "maximum": 1.0, "minimum": 0.0, "title": "Confidence", "type": "number"}}, "required": ["action", "confidence"]}
```

Эту инструкцию мы вставляем в промпт, чтобы модель точно знала, как форматировать ответ.

---

### 3.4. Полный цикл: маршрутизация через PydanticParser

Теперь соберём все компоненты в единую систему. Создадим файл `router_with_parser.py`:

```python
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

# ============================================================================
# 1. Определяем структуру ответа с помощью Pydantic
# ============================================================================

class RouterOutput(BaseModel):
    """Структура ответа маршрутизатора."""
    action: Literal["search", "answer"] = Field(
        description="Действие: search — искать в документах, answer — ответить из знаний"
    )
    confidence: float = Field(
        description="Уверенность в решении (0.0 — 1.0)",
        ge=0.0,
        le=1.0
    )

# ============================================================================
# 2. Создаём парсер
# ============================================================================

parser = PydanticOutputParser(pydantic_object=RouterOutput)

# ============================================================================
# 3. Создаём промпт с инструкциями от парсера
# ============================================================================

prompt = ChatPromptTemplate.from_messages([
    ("system", """
Ты — интеллектуальный маршрутизатор запросов в RAG-системе.

Твоя задача — определить, нужно ли искать информацию в базе знаний, или можно ответить на основе собственных знаний.

База знаний содержит документы о продукте ProjectFlow, финансовые отчёты и историю компании.

Правила принятия решения:
1. Если вопрос требует фактов, цифр или специфической информации из документов → action = "search"
2. Если вопрос общий (математика, философия, погода, юмор) → action = "answer"
3. Если вопрос является уточнением к предыдущему → учитывай контекст (даже если он не сохранён явно)

Верни ответ в строгом формате JSON, следуя инструкциям ниже.

{format_instructions}
"""),
    ("human", "Вопрос: {question}"),
])

# ============================================================================
# 4. Инициализируем LLM с оптимальными параметрами
# ============================================================================

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0.0,      # Детерминированно для маршрутизации
    num_predict=128,      # Короткий ответ — достаточно
    top_p=0.1,            # Ограничиваем выбор для стабильности
)

# ============================================================================
# 5. Собираем цепочку с помощью LCEL
# ============================================================================

chain = prompt | llm | parser

# ============================================================================
# 6. Тестируем на разных вопросах
# ============================================================================

test_questions = [
    "Что такое ProjectFlow?",
    "Какая выручка компании в первом квартале?",
    "Сколько будет 2+2?",
    "Как работает гравитация?",
]

print("=" * 60)
print("ТЕСТИРОВАНИЕ МАРШРУТИЗАТОРА С PYDANTICPARSER")
print("=" * 60)

for question in test_questions:
    print(f"\n{'=' * 60}")
    print(f"Вопрос: {question}")
    print('-' * 60)
    
    try:
        # Вызываем цепочку — результат сразу будет объектом RouterOutput
        result = chain.invoke({
            "question": question,
            "format_instructions": parser.get_format_instructions()
        })
        
        # result — уже объект RouterOutput с валидированными полями!
        print(f"✅ Решение: {result.action}")
        print(f"   Уверенность: {result.confidence:.2f}")
        print(f"   Тип результата: {type(result)}")
        print(f"   Поля: action={result.action}, confidence={result.confidence}")
        
    except Exception as e:
        print(f"❌ Ошибка парсинга: {e}")
        print("   (Это защита от невалидного ответа модели)")
```

#### Реальный вывод при запуске

Вот что мы видим при выполнении скрипта в терминале:

```
(.venv) PS D:\Science\AI_Agent_Demo> python router_with_parser.py

============================================================
ТЕСТИРОВАНИЕ МАРШРУТИЗАТОРА С PYDANTICPARSER
============================================================

============================================================
Вопрос: Что такое ProjectFlow?
------------------------------------------------------------
✅ Решение: search
   Уверенность: 0.90
   Тип результата: <class '__main__.RouterOutput'>
   Поля: action=search, confidence=0.9

============================================================
Вопрос: Какая выручка компании в первом квартале?
------------------------------------------------------------
✅ Решение: search
   Уверенность: 0.90
   Тип результата: <class '__main__.RouterOutput'>
   Поля: action=search, confidence=0.9

============================================================
Вопрос: Сколько будет 2+2?
------------------------------------------------------------
✅ Решение: answer
   Уверенность: 1.00
   Тип результата: <class '__main__.RouterOutput'>
   Поля: action=answer, confidence=1.0

============================================================
Вопрос: Как работает гравитация?
------------------------------------------------------------
✅ Решение: answer
   Уверенность: 1.00
   Тип результата: <class '__main__.RouterOutput'>
   Поля: action=answer, confidence=1.0
(.venv) PS D:\Science\AI_Agent_Demo>
```

**Анализ результатов:**

- Вопросы о ProjectFlow и выручке корректно направляются на поиск (`search`) с уверенностью 0.90.
- Общие вопросы (математика, физика) направляются на ответ из знаний (`answer`) с максимальной уверенностью 1.00.
- Все результаты автоматически валидированы и представлены как объекты Python.

---

### 3.5. Сравнение с ручным подходом из Лекции 6.2

Чтобы оценить преимущества LangChain, давайте сравним код маршрутизатора из Лекции 6.2 и новой версии с PydanticParser.

#### Лекция 6.2 (ручной парсинг JSON)

```python
import json
import requests

def router_manual(question):
    prompt = f"""
    Ты — маршрутизатор. Верни JSON: {{"action": "search" или "answer", "confidence": 0.0-1.0}}
    Вопрос: {question}
    JSON:
    """
    
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "qwen2.5:3b",
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.0}
            },
            timeout=30
        )
        raw_text = response.json().get("response", "")
    except Exception as e:
        print(f"Ошибка запроса: {e}")
        return "search", 0.5  # Fallback
    
    try:
        # Ищем JSON в ответе
        start = raw_text.find('{')
        end = raw_text.rfind('}') + 1
        
        if start == -1 or end == 0:
            raise ValueError("JSON не найден")
        
        json_str = raw_text[start:end]
        data = json.loads(json_str)
        
        action = data.get("action", "search")
        confidence = data.get("confidence", 0.5)
        
        # Валидация вручную
        if action not in ["search", "answer"]:
            action = "search"
        if not (0.0 <= confidence <= 1.0):
            confidence = 0.5
            
        return action, confidence
        
    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f"Ошибка парсинга: {e}")
        return "search", 0.5  # Fallback
```

#### Лекция 6.3 (LangChain с PydanticParser)

```python
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

# Определяем структуру
class RouterOutput(BaseModel):
    action: Literal["search", "answer"] = Field(description="Действие")
    confidence: float = Field(description="Уверенность", ge=0.0, le=1.0)

# Создаём парсер
parser = PydanticOutputParser(pydantic_object=RouterOutput)

# Создаём промпт
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — маршрутизатор. {format_instructions}"),
    ("human", "Вопрос: {question}"),
])

# Инициализируем LLM
llm = ChatOllama(model="qwen2.5:3b", temperature=0.0)

# Собираем цепочку
chain = prompt | llm | parser

# Вызываем
result = chain.invoke({
    "question": "Что такое ProjectFlow?",
    "format_instructions": parser.get_format_instructions()
})

# result — готовый объект RouterOutput!
print(result.action)      # "search"
print(result.confidence)  # 0.9
```

#### Сравнительная таблица

| Аспект | Лекция 6.2 (ручной) | Лекция 6.3 (LangChain) |
|--------|---------------------|----------------------|
| **Объём кода** | 40+ строк | 15 строк |
| **Обработка ошибок** | Ручная (`try/except` для запроса, парсинга, валидации) | Автоматическая (встроена в парсер) |
| **Валидация** | Ручная (проверка полей) | Автоматическая (Pydantic) |
| **Типизация** | Нет (словари) | Есть (объекты Pydantic) |
| **Документация** | Нет | Автоматическая через `Field(description=...)` |
| **Гибкость** | Низкая (правка логики во многих местах) | Высокая (правка одной модели) |
| **Интеграция с цепочками** | Нет | Полная (LCEL) |
| **Повторное использование** | Копипаст | Один класс — много сценариев |
| **Надёжность** | Средняя (зависит от качества ручного парсинга) | Высокая (строгая валидация) |

---

### 3.6. Дополнительные возможности PydanticOutputParser

#### Вложенные структуры

Pydantic позволяет создавать сложные вложенные структуры:

```python
from pydantic import BaseModel
from typing import List

class Document(BaseModel):
    title: str
    content: str
    relevance_score: float

class SearchResult(BaseModel):
    query: str
    documents: List[Document]
    total_count: int

parser = PydanticOutputParser(pydantic_object=SearchResult)
```

#### Кастомные валидаторы

Можно добавить собственные проверки:

```python
from pydantic import BaseModel, validator

class RouterOutput(BaseModel):
    action: str
    confidence: float
    
    @validator('confidence')
    def validate_confidence(cls, v):
        if not (0.0 <= v <= 1.0):
            raise ValueError(f'confidence must be between 0 and 1, got {v}')
        return v
```

#### Обработка ошибок парсинга

LangChain предоставляет механизм для обработки ошибок парсинга:

```python
from langchain_core.output_parsers import OutputFixingParser

# Создаём парсер, который может исправлять ошибки
fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm)

# Используем в цепочке
chain = prompt | llm | fixing_parser
```

---

## Краткий итог Тема 3

- Мы освоили **ChatPromptTemplate** — инструмент для создания структурированных промптов с разделением ролей (system, human, assistant). Это позволяет отделить логику от представления и делает код более поддерживаемым.

- Научились использовать **PydanticOutputParser** — автоматический парсинг ответов в Python-объекты с валидацией. Это полностью исключает ручной парсинг JSON и обработку ошибок.

- Создали полноценный маршрутизатор, который возвращает структурированный ответ без единой строки `json.loads()` или `try/except` для парсинга.

- Увидели, как код стал **чище, короче и надёжнее** по сравнению с ручным подходом из Лекции 6.2. Объём кода сократился в 2–3 раза, а надёжность выросла благодаря автоматической валидации.

- Изучили дополнительные возможности Pydantic: вложенные структуры, кастомные валидаторы и обработку ошибок парсинга.

Теперь у нас есть все необходимые кирпичики для сборки RAG-цепочки:
- **LLM** — `ChatOllama` для вызова модели.
- **Промпты** — `ChatPromptTemplate` для структурированных шаблонов.
- **Парсеры** — `PydanticOutputParser` для автоматической валидации и типизации.

В следующей теме мы объединим их с помощью LCEL и добавим ретривер для поиска в документах, создав полноценную RAG-систему на LangChain. Оставайтесь с нами!


## Тема 4. Создание цепочки RAG с помощью LCEL

В предыдущих темах мы освоили отдельные компоненты LangChain: вызов LLM через `ChatOllama`, создание промптов с `ChatPromptTemplate` и автоматический парсинг ответов через `PydanticOutputParser`. Теперь пришло время объединить их в единую цепочку (chain), которая будет выполнять полноценный RAG-пайплайн.

Напомню, что в Лекции 6.2 мы вручную прописывали каждый шаг: загружали документы, разбивали на чанки, индексировали в Chroma, затем для каждого вопроса выполняли поиск, формировали промпт с контекстом, отправляли HTTP-запрос к Ollama и парсили ответ. Весь код занимал около 150 строк только для основной логики, не считая загрузки документов и чанкинга.

**С LangChain мы сделаем то же самое примерно в 15 строках**, при этом код станет декларативным, легко читаемым и расширяемым. Более того, мы получим возможность легко менять компоненты системы (модель, векторную базу, стратегию поиска) без переписывания всей логики.

---

### 4.1. Загрузка существующей векторной базы Chroma через LangChain

В Лекции 6.2 мы создали векторную базу Chroma в папке `./chroma_db`, используя чистый Python и библиотеку `chromadb`. Теперь мы подключим эту базу через LangChain, используя готовую интеграцию `langchain-chroma`. Это позволит нам не пересоздавать индекс, а сразу использовать уже существующие эмбеддинги, что экономит время и вычислительные ресурсы.

#### 4.1.1. Установка необходимых пакетов

Убедимся, что все нужные пакеты установлены:

```bash
pip install langchain-chroma sentence-transformers langchain-huggingface
```

| Пакет | Назначение |
|-------|------------|
| `langchain-chroma` | Интеграция LangChain с Chroma |
| `sentence-transformers` | Модель для получения эмбеддингов |
| `langchain-huggingface` | Современный адаптер для эмбеддингов (рекомендуется вместо устаревшего `langchain-community`) |

#### 4.1.2. Инициализация модели эмбеддингов

Критически важно: модель эмбеддингов должна быть **строго той же**, что использовалась при создании базы. Иначе векторы будут несовместимы, и поиск даст бессмысленные результаты. В нашем случае это `all-MiniLM-L6-v2` — лёгкая модель с размерностью эмбеддингов 384, хорошо работающая с русским и английским текстами.

**Важно:** ранее мы использовали `SentenceTransformerEmbeddings` из `langchain_community`, но этот модуль помечен как устаревший. Вместо него рекомендуется использовать `HuggingFaceEmbeddings` из `langchain_huggingface`:

```python
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
```

#### 4.1.3. Загрузка векторного хранилища

Класс `Chroma` из `langchain_chroma` предоставляет удобный интерфейс для подключения к существующей базе. **Очень важно указать правильное имя коллекции**, иначе Chroma создаст пустую коллекцию по умолчанию (`"langchain"`) и вы не увидите документов.

В Лекции 6.2 мы создавали коллекцию с именем `"rag_docs"`. Поэтому при загрузке нужно явно передать это имя:

```python
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="./chroma_db",          # Путь к папке с базой
    embedding_function=embedding_model,       # Модель для преобразования текста в векторы
    collection_name="rag_docs"                # Имя коллекции — КЛЮЧЕВОЙ ПАРАМЕТР!
)
```

**Важные параметры класса `Chroma`:**

| Параметр | Описание |
|----------|----------|
| `persist_directory` | Путь к папке, где хранятся данные Chroma. Если папка существует, база загружается. Если нет — создаётся новая. |
| `embedding_function` | Объект, реализующий методы `embed_query` и `embed_documents`. Может быть моделью LangChain или простой функцией. |
| `collection_name` | Имя коллекции в Chroma (по умолчанию `"langchain"`). Должно совпадать с тем, что использовалось при создании базы. |

**Проверка загрузки:**

```python
print(f"✅ База загружена. Количество документов: {vectorstore._collection.count()}")
```

Если вы видите число больше нуля — база успешно подключена. Если ноль — вероятно, указано неверное имя коллекции или путь к папке.

---

### 4.2. Создание ретривера — интерфейса для поиска

Векторное хранилище Chroma предоставляет метод `as_retriever()`, который превращает его в объект-ретривер, совместимый с интерфейсом LangChain. Ретривер умеет принимать запрос (строку) и возвращать список документов (объектов `Document`), найденных по семантической близости.

#### 4.2.1. Что такое ретривер в LangChain?

Ретривер — это абстракция, которая реализует метод `invoke(query)` и возвращает список документов. Он скрывает детали реализации поиска: это может быть векторный поиск в Chroma, полнотекстовый поиск в Elasticsearch, гибридный поиск или даже вызов внешнего API.

```python
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}  # возвращать топ-3 чанка
)
```

**Параметры `search_kwargs`:**

| Параметр | Описание | Пример |
|----------|----------|--------|
| `k` | Количество возвращаемых документов | `3` |
| `score_threshold` | Минимальное значение косинусного расстояния | `0.5` |
| `filter` | Фильтр по метаданным (например, только из определённого файла) | `{"source": "user_guide.pdf"}` |

#### 4.2.2. Использование ретривера

Теперь мы можем вызывать `retriever.invoke("текст вопроса")` и получать список найденных документов. Это полностью заменяет нашу ручную функцию `search_chunks` из Лекции 6.2.

```python
docs = retriever.invoke("Что такое ProjectFlow?")
for doc in docs:
    print(doc.page_content[:100])  # первые 100 символов
    print(f"Источник: {doc.metadata.get('source', 'unknown')}")
    print("-" * 40)
```

**Что возвращает ретривер:**

Каждый документ — это объект класса `Document` из `langchain_core.documents`, который содержит:

- `page_content` — текст документа (строка).
- `metadata` — словарь с метаданными (имя файла, путь, номер чанка и т.д.).

Это полностью соответствует структуре, которую мы создавали вручную в Лекции 6.2.

---

### 4.3. Простейшая цепочка: промпт → LLM → парсер (повторение с углублением)

Прежде чем добавлять ретривер, давайте детально разберём базовую цепочку, чтобы понять фундаментальные принципы LCEL.

#### 4.3.1. Компоненты цепочки

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

# 1. LLM
llm = ChatOllama(model="qwen2.5:3b", temperature=0.0)

# 2. Промпт
prompt = ChatPromptTemplate.from_template("Ответь на вопрос: {question}")

# 3. Парсер
parser = StrOutputParser()

# 4. Цепочка
simple_chain = prompt | llm | parser
```

#### 4.3.2. Оператор `|` — композиция через LCEL

Оператор `|` (вертикальная черта) в LangChain означает **композицию**: выход одного компонента подаётся на вход следующего. Это похоже на пайплайны в Unix (`cat file | grep pattern | wc -l`).

**Как это работает внутри:**

Каждый компонент в LangChain реализует метод `invoke(input)`:

- `prompt.invoke({"question": "..."})` → возвращает отформатированное сообщение (или список сообщений) — объект типа `PromptValue`.
- `llm.invoke(prompt_value)` → принимает сообщение, отправляет запрос в Ollama, возвращает объект `AIMessage`.
- `parser.invoke(ai_message)` → извлекает поле `content` из `AIMessage` и возвращает строку.

Когда мы пишем `prompt | llm | parser`, LangChain создаёт объект `RunnableSequence`, который при вызове `invoke` последовательно выполняет каждый шаг, передавая результат дальше.

#### 4.3.3. Вызов цепочки

```python
result = simple_chain.invoke({"question": "Что такое LangChain?"})
print(result)  # это уже строка
```

**Преимущества такого подхода:**

- **Читаемость** — цепочка читается как инструкция: «взять промпт, потом LLM, потом парсер».
- **Тестируемость** — каждое звено можно протестировать отдельно.
- **Гибкость** — легко добавить шаги (например, постобработку или фильтрацию) или заменить один компонент на другой.

---

### 4.4. Встраивание ретривера в цепочку: `RunnablePassthrough` и `assign`

Теперь ключевая задача: мы хотим, чтобы перед формированием промпта выполнялся поиск в базе знаний. Для этого мы должны построить цепочку, которая:

1. Принимает на вход словарь с полем `"question"`.
2. Передаёт вопрос в ретривер, получает список документов, извлекает из них тексты и объединяет в строку `context`.
3. Передаёт в промпт и исходный вопрос, и полученный контекст.
4. Далее — LLM и парсер.

#### 4.4.1. Форматирование документов в строку контекста

Ретривер возвращает список объектов `Document`. Промпт ожидает строку. Поэтому нам нужно преобразовать документы в текст. Обычно мы просто извлекаем содержимое и объединяем через разделитель:

```python
def format_docs(docs):
    return "\n\n---\n\n".join([doc.page_content for doc in docs])
```

**Почему разделитель `---`:** он чётко отделяет один документ от другого, помогая модели понять, что это разные источники информации.

#### 4.4.2. `RunnablePassthrough` — пропуск данных без изменений

`RunnablePassthrough` — это специальный раннабл, который возвращает входные данные без изменений. Он используется как «проводник» в цепочках, когда нам нужно передать что-то дальше, не модифицируя.

```python
from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough()
result = passthrough.invoke({"question": "Что?"})
# result == {"question": "Что?"}  — данные не изменились
```

Но его основная сила — в методе `assign()`, который позволяет добавлять новые поля в словарь, вычисляя их через функции.

#### 4.4.3. `assign` — добавление вычисляемых полей

Метод `assign` принимает словарь, где ключи — имена новых полей, а значения — функции (или раннаблы), которые их вычисляют.

```python
chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    # Теперь словарь имеет поля: {"question": "...", "context": "..."}
    | prompt
    | llm
    | parser
)
```

**Разбор работы `assign`:**

1. На вход подаётся словарь `{"question": "Что такое ProjectFlow?"}`.
2. `assign` вызывает функцию `lambda x: format_docs(retriever.invoke(x["question"]))`, передавая ей входной словарь.
3. Функция извлекает `question`, выполняет поиск в ретривере, форматирует результат и возвращает строку.
4. `assign` добавляет в исходный словарь новое поле `context` с этой строкой.
5. Полученный словарь `{"question": "...", "context": "..."}` передаётся дальше в промпт.

#### 4.4.4. Альтернативный синтаксис (явный словарь)

Вместо `assign` можно использовать явное построение словаря:

```python
chain = (
    {
        "context": lambda x: format_docs(retriever.invoke(x["question"])),
        "question": lambda x: x["question"]
    }
    | prompt
    | llm
    | parser
)
```

Этот вариант более явный, но при добавлении новых полей становится громоздким. `assign` удобнее, когда у нас много полей, которые мы просто пропускаем дальше.

---

### 4.5. Полная RAG-цепочка: собираем всё вместе

Теперь создадим файл `rag_chain.py` с полной реализацией. Мы будем использовать ту же базу данных, что создали в Лекции 6.2.

**Полный код с подробными комментариями и современными импортами:**

```python
# ============================================================================
# Импорты (современные)
# ============================================================================

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ============================================================================
# 1. Загрузка векторной базы Chroma
# ============================================================================

# Модель эмбеддингов — должна быть ТОЧНО ТАКОЙ ЖЕ, как при создании базы!
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Подключение к существующей базе
vectorstore = Chroma(
    persist_directory="./chroma_db",          # Путь к папке с базой
    embedding_function=embedding_model,       # Функция эмбеддингов
    collection_name="rag_docs"                # Имя коллекции — КЛЮЧЕВОЙ ПАРАМЕТР!
)

print(f"✅ База загружена. Количество документов: {vectorstore._collection.count()}")

# ============================================================================
# 2. Создание ретривера
# ============================================================================

# Настраиваем ретривер: возвращаем топ-3 чанка по релевантности
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# ============================================================================
# 3. Функция форматирования документов
# ============================================================================

def format_docs(docs):
    """
    Преобразует список документов в единую строку с разделителями.
    
    Аргументы:
        docs: список объектов Document (с полями page_content и metadata)
    
    Возвращает:
        Строку, где содержимое документов разделено тремя дефисами.
    """
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

# ============================================================================
# 4. Промпт для RAG
# ============================================================================

# Шаблон промпта с системной инструкцией и переменными
prompt = ChatPromptTemplate.from_messages([
    ("system", """
Ты — строгий помощник. Отвечай ТОЛЬКО на основе приведённого КОНТЕКСТА.
Если в контексте нет прямого ответа на вопрос, скажи: "В документах нет информации по этому вопросу."

КОНТЕКСТ:
{context}
"""),
    ("human", "ВОПРОС: {question}\n\nОТВЕТ (только из контекста):"),
])

# ============================================================================
# 5. Инициализация LLM
# ============================================================================

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0.0,      # Детерминированный вывод
    num_predict=512,      # Достаточно для развёрнутого ответа
)

# ============================================================================
# 6. Сборка RAG-цепочки с помощью LCEL
# ============================================================================

rag_chain = (
    # Шаг 1: добавляем поле context, полученное через ретривер
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    # Шаг 2: передаём словарь {"question": ..., "context": ...} в промпт
    | prompt
    # Шаг 3: передаём форматированное сообщение в LLM
    | llm
    # Шаг 4: извлекаем строку из ответа модели
    | StrOutputParser()
)

# ============================================================================
# 7. Тестирование цепочки
# ============================================================================

if __name__ == "__main__":
    # Пример вопроса, на который есть ответ в документах
    question = "Что такое ProjectFlow?"
    print(f"Вопрос: {question}")
    print("-" * 60)
    
    answer = rag_chain.invoke({"question": question})
    print(f"Ответ: {answer}")
    
    print("\n" + "=" * 60)
    
    # Пример вопроса, которого нет в документах
    question2 = "Какая погода завтра?"
    print(f"Вопрос: {question2}")
    print("-" * 60)
    
    answer2 = rag_chain.invoke({"question": question2})
    print(f"Ответ: {answer2}")
```

---

### 4.6. Пример вывода при запуске

После исправления импортов и добавления имени коллекции мы получаем ожидаемый результат:

```
(.venv) PS D:\Science\AI_Agent_Demo> python rag_chain.py
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading weights: 100%|████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 13931.42it/s]
✅ База загружена. Количество документов: 10
Вопрос: Что такое ProjectFlow?
------------------------------------------------------------
Ответ: ProjectFlow — это облачная платформа для управления задачами и командной работой. Основные возможности включают создание проектов и подзадач, назначение исполнителей и сроков, отслеживание прогресса через диаграммы Ганта, интеграцию с календарями (Google Calendar, Outlook) и поддержку файлового хранилища (до 50 ГБ на проект).

============================================================
Вопрос: Какая погода завтра?
------------------------------------------------------------
Ответ: В документах нет информации по этому вопросу.
```

**Анализ работы:**

1. Для вопроса о ProjectFlow ретривер нашёл релевантные чанки из `user_guide.pdf`, они были переданы в промпт, и модель дала точный ответ.
2. Для вопроса о погоде ретривер не нашёл релевантных документов (или они не содержали информацию о погоде), и модель корректно сообщила об отсутствии данных.

---

### 4.7. Важные замечания по настройке

В ходе работы могут возникнуть следующие типичные проблемы:

| Проблема | Причина | Решение |
|----------|---------|---------|
| `Количество документов: 0` | Не указано имя коллекции или оно не совпадает с тем, что использовалось при создании базы | Добавьте `collection_name="rag_docs"` при создании `Chroma` |
| `DeprecationWarning` об `langchain-community` | Используется устаревший модуль | Замените `SentenceTransformerEmbeddings` на `HuggingFaceEmbeddings` из `langchain_huggingface` |
| `Warning: You are sending unauthenticated requests to the HF Hub` | Не указан токен для Hugging Face | Это предупреждение, его можно игнорировать, или задать `HF_TOKEN` |
| База находится в другой папке | Путь `persist_directory` указан неверно | Укажите правильный путь, например `"./Part2/chroma_db"` |

---

### 4.8. Сравнение объёма кода: ручной подход vs LangChain

Давайте наглядно сравним, сколько кода потребовалось для реализации RAG-логики в каждом подходе.

#### Лекция 6.2 (ручной подход) — только основная логика

```python
# Примерная структура кода из Лекции 6.2

def search_chunks(collection, query, top_k=3):
    # 15 строк: эмбеддинг + поиск в Chroma + форматирование
    pass

def answer_question(question, collection, memory):
    # 1. Маршрутизация (20 строк)
    # 2. Если search — поиск (10 строк)
    # 3. Формирование промпта с контекстом (10 строк)
    # 4. HTTP-запрос к Ollama (15 строк)
    # 5. Парсинг ответа (10 строк)
    # 6. Сохранение в память (5 строк)
    # 7. Логирование (5 строк)
    # Итого: ~75 строк на основной метод
    pass

# Итого: около 150 строк на RAG-логику (без учёта загрузки и чанкинга)
```

#### Лекция 6.3 (LangChain) — вся RAG-цепочка

```python
# Вся логика умещается в 15 строк

rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | prompt
    | llm
    | StrOutputParser()
)
```

**Что сократилось:**

| Компонент | Ручной подход | LangChain |
|-----------|---------------|-----------|
| Поиск в Chroma | 15 строк (эмбеддинг + запрос) | 1 строка (`retriever.invoke`) |
| Формирование промпта | 5–10 строк (f-строки) | Шаблон (вынесен отдельно) |
| Вызов LLM | 15 строк (requests.post + обработка) | 0 строк (встроено в LLM) |
| Парсинг ответа | 10 строк (json.loads + try/except) | 0 строк (StrOutputParser) |
| Связывание шагов | 20+ строк (ручные вызовы) | 5 строк (оператор `\|`) |

**Итог:** код стал не только короче, но и прозрачнее — мы видим всю логику как единый пайплайн.

---

### 4.9. Что даёт нам LCEL и почему это важно

| Аспект | Описание |
|--------|----------|
| **Декларативность** | Мы описываем **что** делаем, а не **как**. Код становится самодокументируемым. |
| **Гибкость** | Легко заменить компонент: например, поменять `ChatOllama` на `ChatOpenAI` или `retriever` на другой источник. |
| **Тестируемость** | Каждый раннабл можно протестировать изолированно, подставляя mock-объекты. |
| **Масштабируемость** | Цепочки можно комбинировать, вкладывать друг в друга, создавать сложные маршруты. |
| **Надёжность** | Встроенная обработка ошибок и типизация через Pydantic уменьшают число багов. |
| **Поддержка** | Новый разработчик быстрее поймёт логику, глядя на цепочку, чем на 150 строк императивного кода. |

---

### 4.10. Как работает цепочка «под капотом»

Чтобы глубже понять, что происходит при вызове `rag_chain.invoke(...)`, давайте заглянем внутрь:

1. **Вызов `invoke`** с входным словарём.
2. **`RunnablePassthrough.assign`** — создаёт новый словарь, добавляя поле `context`. Для этого он вызывает переданную функцию, которая внутри:
   - Вызывает `retriever.invoke(question)`, что выполняет поиск в Chroma.
   - `retriever` вычисляет эмбеддинг вопроса через `embedding_model.embed_query()`, затем отправляет запрос в Chroma и возвращает топ-3 документа.
   - `format_docs` извлекает `page_content` из каждого документа и склеивает их.
3. **Полученный словарь** с полями `question` и `context` передаётся в `prompt`.
4. **`prompt`** форматирует шаблон, подставляя значения, и возвращает список сообщений (SystemMessage + HumanMessage).
5. **`llm`** отправляет сообщения в Ollama, получает ответ в виде `AIMessage`.
6. **`StrOutputParser`** извлекает поле `content` из `AIMessage`.
7. **Возвращается** итоговая строка.

---

## Краткий итог Тема 4

- Мы научились подключать существующую векторную базу Chroma через LangChain с помощью класса `Chroma`, обязательно указывая `collection_name="rag_docs"` и используя современный класс эмбеддингов `HuggingFaceEmbeddings` вместо устаревшего `SentenceTransformerEmbeddings`.

- Создали ретривер (`vectorstore.as_retriever`), который по запросу возвращает топ-K документов, полностью заменив нашу ручную функцию `search_chunks`.

- Освоили оператор `|` в LCEL для композиции компонентов — теперь логика собирается как пайплайн, где выход одного шага становится входом для следующего.

- Использовали `RunnablePassthrough.assign` для добавления в пайплайн поля `context`, полученного из ретривера. Это позволило нам объединить поиск и генерацию в единую цепочку.

- Собрали полноценную RAG-цепочку, которая по вопросу находит релевантные документы, формирует промпт, отправляет в LLM и возвращает ответ.

- Сравнили объём кода: **15 строк** в LangChain против **~150 строк** в ручной реализации из Лекции 6.2. При этом код стал более читаемым, гибким и надёжным.

- Разобрали, как цепочка работает «под капотом», чтобы вы понимали внутренние механизмы, а не просто копировали код.

---

**Мы получили элегантный RAG-пайплайн, который легко читается и расширяется. Но пока он “без памяти” — каждый вопрос обрабатывается изолированно, без учёта истории диалога. В следующей теме мы добавим память, чтобы агент мог поддерживать связный разговор, а также рассмотрим, как интегрировать маршрутизацию, чтобы система сама решала, когда искать в документах, а когда отвечать из знаний. Оставайтесь с нами!**

## Тема 5. Добавление памяти: история диалога

В Лекции 6.2 мы реализовали память вручную — создали класс `ConversationMemory`, который хранил историю в списке словарей и подставлял её в промпт. Это работало, но требовало дополнительного кода: управления списком, обрезки истории, форматирования в строку. Каждый раз нам приходилось вручную извлекать историю, преобразовывать её в текст и вставлять в промпт. Это создавало много шаблонного кода и усложняло поддержку.

**LangChain предоставляет готовые решения для управления памятью.** Однако важно учитывать, что экосистема активно развивается. На 2026 год:

- Модуль `langchain-community` официально устарел и заархивирован; вместо него используются специализированные пакеты интеграций (`langchain-chroma`, `langchain-huggingface`, `langchain-ollama`).
- Класс `ChatMessageHistory` из `langchain_community` заменён на `InMemoryChatMessageHistory` из `langchain_core.chat_history`.
- `RunnableWithMessageHistory` пока работает, но для продакшена рекомендуется LangGraph с встроенной персистентностью.

В этой теме мы реализуем память с использованием **современных актуальных компонентов**, которые не вызывают предупреждений об устаревании.

---

### 5.1. Концепция памяти в LangChain

#### 5.1.1. Основные компоненты памяти

| Компонент | Назначение | Пример использования |
|-----------|------------|---------------------|
| `InMemoryChatMessageHistory` | Хранит список сообщений в сессии (в оперативной памяти) | `history = InMemoryChatMessageHistory()` |
| `RunnableWithMessageHistory` | Обёртка, автоматически подставляющая историю в цепочку и сохраняющая новые сообщения | `rag_with_memory = RunnableWithMessageHistory(...)` |
| `MessagesPlaceholder` | Место в промпте для вставки списка сообщений | `MessagesPlaceholder(variable_name="history")` |
| `get_session_history` | Функция, возвращающая объект истории по `session_id` | `def get_session_history(sid): return store[sid]` |

#### 5.1.2. Как это работает

1. **Хранение:** `InMemoryChatMessageHistory` хранит список объектов `BaseMessage` (HumanMessage, AIMessage).
2. **Извлечение:** `RunnableWithMessageHistory` при вызове `invoke` обращается к `get_session_history(session_id)` и получает объект истории.
3. **Подстановка:** История (как список сообщений) подставляется в промпт через `MessagesPlaceholder`.
4. **Сохранение:** После выполнения цепочки новые сообщения (вопрос пользователя и ответ ассистента) автоматически добавляются в историю.

Такой подход полностью избавляет от ручного управления списком и форматирования.

#### 5.1.3. Почему `InMemoryChatMessageHistory`?

В старых версиях LangChain использовался модуль `langchain.memory` с классами `ConversationBufferMemory`, `ConversationBufferWindowMemory` и т.д. В новых версиях эти классы перенесены в `langchain_community.memory`, а сам пакет `langchain-community` помечен как устаревший. Вместо этого рекомендуется использовать `InMemoryChatMessageHistory` из `langchain_core.chat_history`, который не требует внешних зависимостей и полностью совместим с `RunnableWithMessageHistory`.

**Актуальные импорты:**

```python
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
```

---

### 5.2. Реализация памяти в коде

#### 5.2.1. Создание хранилища сессий

Хранилище сессий — это словарь, который сопоставляет `session_id` (идентификатор пользователя или сессии) с объектом `InMemoryChatMessageHistory`. В реальных приложениях вместо словаря в памяти могут использоваться базы данных (Redis, PostgreSQL), но для демонстрации мы используем простой словарь.

```python
from langchain_core.chat_history import InMemoryChatMessageHistory
from typing import Dict

# Хранилище сессий: session_id → InMemoryChatMessageHistory
session_store: Dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """
    Возвращает объект истории для данной сессии.
    Если сессии нет — создаёт новую.
    
    Аргументы:
        session_id: уникальный идентификатор сессии (пользователя)
    
    Возвращает:
        InMemoryChatMessageHistory — объект с историей сообщений
    """
    if session_id not in session_store:
        # Создаём новую историю для сессии
        session_store[session_id] = InMemoryChatMessageHistory()
        print(f"🆕 Создана новая сессия: {session_id}")
    return session_store[session_id]
```

#### 5.2.2. Промпт с `MessagesPlaceholder`

`MessagesPlaceholder` — это компонент, который вставляет список сообщений в нужное место промпта. Это гораздо удобнее, чем ручное форматирование строк, потому что:

1. Сообщения сохраняют свою структуру (роль, содержимое).
2. Модель получает структурированный диалог, что улучшает качество ответов.
3. Не нужно экранировать спецсимволы.

```python
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

rag_prompt = ChatPromptTemplate.from_messages([
    # Системное сообщение с инструкцией
    ("system", """
Ты — строгий помощник. Отвечай ТОЛЬКО на основе приведённого КОНТЕКСТА.
Если в контексте нет прямого ответа на вопрос, скажи: "В документах нет информации по этому вопросу."

КОНТЕКСТ:
{context}
"""),
    # ВСТАВКА ИСТОРИИ ДИАЛОГА
    MessagesPlaceholder(variable_name="history"),
    # Текущий вопрос пользователя
    ("human", "ВОПРОС: {question}\n\nОТВЕТ (только из контекста):"),
])
```

**Что здесь происходит:**

1. `MessagesPlaceholder` будет заменён на список сообщений из истории.
2. История вставляется **между** системной инструкцией и текущим вопросом.
3. Это создаёт правильную структуру диалога: система → история → новый вопрос.

#### 5.2.3. Обёртка `RunnableWithMessageHistory`

`RunnableWithMessageHistory` — это обёртка, которая принимает базовую цепочку и автоматически управляет историей.

```python
from langchain_core.runnables.history import RunnableWithMessageHistory

rag_with_memory = RunnableWithMessageHistory(
    runnable=rag_chain,                     # Базовая RAG-цепочка
    get_session_history=get_session_history, # Функция получения истории
    input_messages_key="question",          # Ключ, по которому передаётся вопрос
    history_messages_key="history"          # Ключ для истории в промпте
)
```

**Параметры `RunnableWithMessageHistory`:**

| Параметр | Описание |
|----------|----------|
| `runnable` | Базовая цепочка, которую мы оборачиваем |
| `get_session_history` | Функция, возвращающая `InMemoryChatMessageHistory` по `session_id` |
| `input_messages_key` | Ключ в словаре, содержащий сообщение пользователя |
| `history_messages_key` | Ключ, под которым история передаётся в цепочку (должен совпадать с `MessagesPlaceholder`) |
| `output_messages_key` | (опционально) Ключ, под которым сохраняется ответ ассистента |

#### 5.2.4. Вызов цепочки с памятью

```python
# Вызов с указанием session_id
response = rag_with_memory.invoke(
    {"question": "Что такое ProjectFlow?"},
    config={"configurable": {"session_id": "user_123"}}
)
```

**Что происходит при вызове:**

1. `RunnableWithMessageHistory` извлекает историю по `session_id` через `get_session_history`.
2. Подставляет историю в промпт через `MessagesPlaceholder`.
3. Выполняет базовую цепочку (`rag_chain`).
4. Автоматически добавляет в историю сообщение пользователя и ответ ассистента.

---

### 5.3. Вспомогательная функция для получения истории в текстовом виде

Для маршрутизации нам может потребоваться история в виде текста (а не списка объектов). Напишем функцию, которая преобразует `InMemoryChatMessageHistory` в строку.

```python
def get_history_text(session_id: str) -> str:
    """
    Преобразует историю сессии в текстовый формат для маршрутизатора.
    
    Аргументы:
        session_id: идентификатор сессии
    
    Возвращает:
        Строку с историей диалога
    """
    history = get_session_history(session_id)
    messages = history.messages
    
    if not messages:
        return "Нет предыдущих сообщений"
    
    lines = []
    for msg in messages:
        if msg.type == "human":
            lines.append(f"Пользователь: {msg.content}")
        elif msg.type == "ai":
            lines.append(f"Ассистент: {msg.content}")
    
    return "\n".join(lines)
```

---

### 5.4. Ограничение длины истории

В долгих диалогах история может стать слишком длинной и не поместиться в контекстное окно модели. Для решения этой проблемы можно реализовать обрезку истории.

```python
def get_session_history_trimmed(session_id: str, max_pairs: int = 5) -> InMemoryChatMessageHistory:
    """
    Возвращает историю сессии, обрезанную до последних max_pairs сообщений.
    """
    history = get_session_history(session_id)
    messages = history.messages
    
    # Оставляем только последние max_pairs * 2 сообщений (пар)
    if len(messages) > max_pairs * 2:
        trimmed = InMemoryChatMessageHistory()
        for msg in messages[-max_pairs * 2:]:
            if msg.type == "human":
                trimmed.add_user_message(msg.content)
            elif msg.type == "ai":
                trimmed.add_ai_message(msg.content)
        return trimmed
    
    return history
```

Этот подход можно использовать в `RunnableWithMessageHistory`, передавая обрезанную историю.

---

### 5.5. Полный код агента с памятью и маршрутизацией (актуальный на 2026)

Ниже представлен полный код, который объединяет все компоненты: загрузку векторной базы, RAG-цепочку, маршрутизацию, память и вспомогательные функции. Код использует актуальные импорты и учитывает предупреждения об устаревании.

```python
# ============================================================================
# ИМПОРТЫ (актуальные на 2026 год)
# ============================================================================

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory  # <--- НОВЫЙ ИМПОРТ
from pydantic import BaseModel, Field
from typing import Literal, Dict

# ============================================================================
# 1. МОДЕЛЬ ДЛЯ МАРШРУТИЗАЦИИ
# ============================================================================

class RouterOutput(BaseModel):
    """Структура ответа маршрутизатора."""
    action: Literal["search", "answer"] = Field(
        description="Действие: search — искать в документах, answer — ответить из знаний"
    )
    confidence: float = Field(
        description="Уверенность в решении (0.0 — 1.0)",
        ge=0.0,
        le=1.0
    )

parser = PydanticOutputParser(pydantic_object=RouterOutput)

# ============================================================================
# 2. ЗАГРУЗКА ВЕКТОРНОЙ БАЗЫ
# ============================================================================

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model,
    collection_name="rag_docs"
)

print(f"✅ База загружена. Количество документов: {vectorstore._collection.count()}")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

# ============================================================================
# 3. RAG-ПРОМПТ С ПАМЯТЬЮ
# ============================================================================

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Ты — строгий помощник. Отвечай ТОЛЬКО на основе приведённого КОНТЕКСТА.
Если в контексте нет прямого ответа на вопрос, скажи: "В документах нет информации по этому вопросу."

КОНТЕКСТ:
{context}
"""),
    MessagesPlaceholder(variable_name="history"),
    ("human", "ВОПРОС: {question}\n\nОТВЕТ (только из контекста):"),
])

# ============================================================================
# 4. LLM
# ============================================================================

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0.0,
    num_predict=512,
)

router_llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0.0,
    num_predict=128,
)

# ============================================================================
# 5. RAG-ЦЕПОЧКА
# ============================================================================

rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | rag_prompt
    | llm
    | StrOutputParser()
)

# ============================================================================
# 6. ЦЕПОЧКА МАРШРУТИЗАЦИИ (УЛУЧШЕННАЯ)
# ============================================================================

COLLECTION_DESCRIPTION = """
База знаний содержит следующие документы:
1. Руководство пользователя ProjectFlow — облачной платформы для управления проектами.
2. Финансовый отчёт компании "Техно-Инновации" за первый квартал 2026 года.
3. История компании "Техно-Инновации".

ВАЖНО: ProjectFlow — ЭТО ПРОДУКТ, ОПИСАННЫЙ В ДОКУМЕНТАХ, а не продукт Microsoft.
"""

router_prompt = ChatPromptTemplate.from_messages([
    ("system", f"""
Ты — интеллектуальный маршрутизатор запросов в RAG-системе.

{COLLECTION_DESCRIPTION}

Правила принятия решения (в порядке приоритета):

1. Если вопрос содержит математические операции (сложение, вычитание, умножение, деление) → action = "answer"

2. Если вопрос касается ProjectFlow, его тарифов, выручки или истории компании → action = "search"

3. Если вопрос требует фактов, цифр или специфической информации из документов → action = "search"

4. Если вопрос общий (философия, погода, юмор, общие знания) → action = "answer"

5. Если вопрос является уточнением к предыдущему, и предыдущий вопрос требовал поиска → action = "search"

ИСТОРИЯ ДИАЛОГА (для понимания контекста):
{{history}}

Верни ответ в строгом формате JSON.

{{format_instructions}}
"""),
    ("human", "ВОПРОС: {{question}}"),
])

router_chain = router_prompt | router_llm | parser

# ============================================================================
# 7. ХРАНИЛИЩЕ СЕССИЙ (С InMemoryChatMessageHistory)
# ============================================================================

session_store: Dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Возвращает историю для данной сессии."""
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
        print(f"🆕 Создана новая сессия: {session_id}")
    return session_store[session_id]

# ============================================================================
# 8. ОБЁРТКА RAG-ЦЕПОЧКИ С ПАМЯТЬЮ
# ============================================================================

rag_with_memory = RunnableWithMessageHistory(
    runnable=rag_chain,
    get_session_history=get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

# ============================================================================
# 9. ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ
# ============================================================================

def get_history_text(session_id: str) -> str:
    """Преобразует историю в текстовый формат."""
    history = get_session_history(session_id)
    messages = history.messages
    
    if not messages:
        return "Нет предыдущих сообщений"
    
    lines = []
    for msg in messages:
        if msg.type == "human":
            lines.append(f"Пользователь: {msg.content}")
        elif msg.type == "ai":
            lines.append(f"Ассистент: {msg.content}")
    
    return "\n".join(lines)

# ============================================================================
# 10. ОСНОВНАЯ ФУНКЦИЯ
# ============================================================================

def answer_with_routing(question: str, session_id: str = "default") -> str:
    """Обрабатывает вопрос с маршрутизацией и памятью."""
    history_text = get_history_text(session_id)
    
    try:
        route = router_chain.invoke({
            "question": question,
            "history": history_text,
            "format_instructions": parser.get_format_instructions()
        })
        print(f"🎯 Маршрутизация: {route.action} (уверенность: {route.confidence:.2f})")
    except Exception as e:
        print(f"⚠️ Ошибка маршрутизации: {e}, используем search")
        route = RouterOutput(action="search", confidence=0.5)
    
    if route.action == "search":
        return rag_with_memory.invoke(
            {"question": question},
            config={"configurable": {"session_id": session_id}}
        )
    else:
        return answer_from_knowledge(question, history_text, session_id)

def answer_from_knowledge(question: str, history: str, session_id: str) -> str:
    """Отвечает из общих знаний."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Ответь на вопрос, используя свои знания. Будь кратким."),
        ("human", "ВОПРОС: {question}"),
    ])
    
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"question": question})
    
    history_obj = get_session_history(session_id)
    history_obj.add_user_message(question)
    history_obj.add_ai_message(response)
    
    return response

# ============================================================================
# 11. ТЕСТИРОВАНИЕ
# ============================================================================

if __name__ == "__main__":
    session_id = "user_123"
    
    questions = [
        "Что такое ProjectFlow?",
        "А какие у неё тарифы?",
        "Сколько будет 2+2?",
        "Какую выручку компания получила в первом квартале?",
        "А сколько это в долларах?"
    ]
    
    for question in questions:
        print(f"\n{'=' * 60}")
        print(f"Вопрос: {question}")
        print('-' * 60)
        
        response = answer_with_routing(question, session_id)
        print(f"Ответ: {response}")
```

---

### 5.6. Результаты тестирования и анализ

После запуска приведённого кода с корректно заполненной базой Chroma (содержащей документы из Лекции 6.2) мы получили следующий вывод:

```
✅ База загружена. Количество документов: 10
🆕 Создана новая сессия: user_123

============================================================
Вопрос: Что такое ProjectFlow?
------------------------------------------------------------
🎯 Маршрутизация: search (уверенность: 0.90)
Ответ: ProjectFlow — это облачная платформа для управления задачами и командной работой. Основные возможности включают создание проектов и подзадач, назначение исполнителей и сроков, отслеживание прогресса через диаграммы Ганта, интеграцию с календарями (Google Calendar, Outlook) и поддержку файлового хранилища (до 50 ГБ на проект).

============================================================
Вопрос: А какие у неё тарифы?
------------------------------------------------------------
🎯 Маршрутизация: search (уверенность: 1.00)
Ответ: тарифа: Starter (бесплатно, до 5 пользователей), Pro (9.99$/мес, неограниченно пользователей), Enterprise (индивидуально).

============================================================
Вопрос: Сколько будет 2+2?
------------------------------------------------------------
🎯 Маршрутизация: search (уверенность: 1.00)
Ответ: 4

============================================================
Вопрос: Какую выручку компания получила в первом квартале?
------------------------------------------------------------
🎯 Маршрутизация: search (уверенность: 0.90)
Ответ: В документах нет информации по этому вопросу.

============================================================
Вопрос: А сколько это в долларах?
------------------------------------------------------------
🎯 Маршрутизация: search (уверенность: 1.00)
Ответ: В документах нет информации по этому вопросу.
```

**Анализ результатов:**

1. **Первый вопрос о ProjectFlow** классифицирован как `search` (уверенность 0.90) — это означает, что улучшенный промпт маршрутизатора с явным описанием базы знаний сработал. Ответ получен из документов, что подтверждает корректную работу ретривера и RAG-цепочки.

2. **Второй вопрос о тарифах** также правильно направлен на поиск, и ответ извлечён из документов.

3. **Математический вопрос `2+2`** ошибочно классифицирован как `search` (уверенность 1.00), хотя должен быть `answer`. Это связано с тем, что правило о математике в промпте маршрутизатора имеет более низкий приоритет, чем другие правила. Для исправления следует перенести математическое правило на первое место и добавить примеры. Однако даже при ошибочной классификации ответ «4» верен, так как модель в RAG-цепочке, вероятно, не нашла контекста и использовала общие знания (хотя инструкция «отвечай ТОЛЬКО на основе контекста» могла бы помешать — но она смогла ответить). Это говорит о том, что механизм fallback работает.

4. **Вопрос о выручке** классифицирован как `search`, но ответ «В документах нет информации». Это указывает на то, что либо документ `q1_report.docx` не был загружен в базу, либо ретривер не нашёл релевантного чанка, либо в чанке не было точной цифры. Проверка содержимого базы и параметров чанкинга может решить проблему.

5. **Вопрос о переводе в доллары** также классифицирован как `search`, и ответ о отсутствии информации корректен, так как в документах нет курса валют.

**Выводы по тестированию:**

- Память работает: агент корректно обрабатывает уточняющие вопросы («А какие у неё тарифы?»), понимая, что «неё» относится к ProjectFlow.
- Маршрутизация улучшена: вопросы о продукте направляются на поиск.
- Есть мелкие недочёты (математика, отсутствие выручки), которые не влияют на общую работоспособность и легко исправляются настройкой промпта или проверкой данных.

---

### 5.7. Сравнение с ручным подходом из Лекции 6.2

| Аспект | Лекция 6.2 (ручной) | Лекция 6.3 (LangChain) |
|--------|---------------------|----------------------|
| **Хранение истории** | Список словарей + ручное управление | `InMemoryChatMessageHistory` + автоматическое управление |
| **Подстановка в промпт** | Ручное форматирование строк | `MessagesPlaceholder` |
| **Управление сессиями** | Ручное (словарь с историей) | `RunnableWithMessageHistory` |
| **Обрезка истории** | Ручная реализация | Можно реализовать через обёртку |
| **Сохранение ответов** | Ручное добавление в список | Автоматическое добавление |
| **Маршрутизация** | Ручной парсинг JSON с `try/except` | `PydanticOutputParser` + чёткий промпт |
| **Объём кода** | ~100 строк на память + маршрутизацию | ~30 строк (без учёта повторяющихся блоков) |
| **Надёжность** | Средняя (зависит от качества парсинга) | Высокая (автоматическая валидация) |

---

## Краткий итог Тема 5

- Мы реализовали **память** с использованием современных компонентов LangChain: `InMemoryChatMessageHistory` и `RunnableWithMessageHistory`, что полностью автоматизирует управление историей диалога.
- **Промпт** теперь содержит `MessagesPlaceholder`, что позволяет вставлять структурированную историю без ручного форматирования.
- **Маршрутизация** учитывает историю диалога, что повышает точность принятия решений.
- **Код** стал значительно чище, компактнее и надёжнее по сравнению с ручной реализацией из Лекции 6.2.
- **Сессии** позволяют вести несколько независимых диалогов одновременно.
- **Автоматическое сохранение** избавляет от необходимости вручную добавлять сообщения в историю.

Теперь наш агент может поддерживать связный диалог, понимать уточняющие вопросы и принимать решения с учётом контекста разговора. Это делает систему более естественной и удобной для пользователя.

> **Примечание:** В продакшене рекомендуется использовать **LangGraph** с встроенной персистентностью вместо `RunnableWithMessageHistory`, так как LangGraph предоставляет более гибкие механизмы управления состоянием и долгосрочной памятью. Однако для учебных целей и простых приложений `RunnableWithMessageHistory` остаётся работоспособным.

## Итоговый код: компактный RAG-агент на LangChain

После изучения всех тем — от вызова LLM до памяти и маршрутизации — давайте соберём всё в **единый компактный модуль**. Ниже представлен класс `LangChainRAG`, который объединяет все ключевые компоненты:

- загрузка векторной базы Chroma и создание ретривера;
- RAG-цепочка с памятью (`RunnableWithMessageHistory`);
- (опционально) маршрутизатор с `PydanticOutputParser` для интеллектуального выбора между поиском и ответом из знаний.

Этот код можно использовать как готовый «движок» для ваших приложений. Он модульный, легко расширяется и заменяет сотни строк ручного кода из Лекции 6.2.

```python
# ============================================================================
# ИТОГОВЫЙ КОД: КОМПАКТНЫЙ RAG-АГЕНТ НА LANGCHAIN (30–40 строк)
# ============================================================================

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from typing import Dict

# 1. Загрузка базы и ретривер
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding_model,
    collection_name="rag_docs"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

# 2. Промпт с историей
prompt = ChatPromptTemplate.from_messages([
    ("system", "Отвечай ТОЛЬКО на основе контекста.\nКОНТЕКСТ: {context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "ВОПРОС: {question}"),
])

# 3. LLM
llm = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=512)

# 4. RAG-цепочка (без памяти)
rag_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Память (хранилище сессий)
session_store: Dict[str, InMemoryChatMessageHistory] = {}
def get_session_history(session_id: str):
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

# 6. Финальная цепочка с памятью
rag_with_memory = RunnableWithMessageHistory(
    runnable=rag_chain,
    get_session_history=get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

# 7. Пример использования
if __name__ == "__main__":
    response = rag_with_memory.invoke(
        {"question": "Что такое ProjectFlow?"},
        config={"configurable": {"session_id": "user_123"}}
    )
    print(response)
```

**Всего 43 строки** (с импортами и пустыми строками) — и у нас полноценный RAG-агент с памятью! Для сравнения, в Лекции 6.2 аналогичный функционал занимал около **150 строк** только для RAG-логики, не считая загрузки документов и чанкинга.

---

## Сравнение с кодом из Лекции 6.2

| Аспект | Лекция 6.2 (ручной код) | Лекция 6.3 (LangChain) |
|--------|-------------------------|------------------------|
| **Объём кода (основная логика)** | ~150 строк | ~40 строк |
| **Управление памятью** | Ручной список словарей + форматирование | `InMemoryChatMessageHistory` + `RunnableWithMessageHistory` |
| **Интеграция с векторной БД** | Ручной запрос к Chroma через `chromadb` | Готовый ретривер через `langchain-chroma` |
| **Формирование промпта** | f-строки в коде | Шаблон `ChatPromptTemplate` |
| **Парсинг ответа** | Ручной `json.loads()` + `try/except` | `StrOutputParser` / `PydanticOutputParser` |
| **Гибкость** | Низкая (замена модели требует правки API) | Высокая (замена через один параметр) |
| **Расширяемость** | Сложно добавить новые инструменты | Легко (LCEL позволяет вставлять новые звенья) |

**Главное достижение:** мы не потеряли понимания «под капотом», но теперь наш код стал **декларативным, модульным и готовым к промышленному использованию**. Мы можем легко:
- заменить `ChatOllama` на `ChatOpenAI` или `ChatAnthropic`,
- подключить другой ретривер (например, Elasticsearch или веб-поиск),
- добавить дополнительные шаги (фильтрацию документов, пост-обработку ответа) — всё это делается простым добавлением звеньев в цепочку.

---

## Заключение Лекции 6.3

Поздравляю! Мы прошли путь от «ручного» RAG на чистых HTTP-запросах и собственном коде до элегантной, модульной системы на LangChain. Вы не просто скопировали готовые решения — вы **поняли каждый компонент** и увидели, как фреймворк абстрагирует рутину, не скрывая сути.

### Что мы сделали за эту лекцию

- **Тема 1:** познакомились с LangChain, его компонентами и LCEL.
- **Тема 2:** освоили вызов LLM через `ChatOllama` и тонкую настройку параметров.
- **Тема 3:** научились создавать структурированные промпты и автоматически парсить ответы с помощью Pydantic.
- **Тема 4:** собрали полноценную RAG-цепочку с ретривером и LCEL — в 15 строк.
- **Тема 5:** добавили память через `InMemoryChatMessageHistory` и `RunnableWithMessageHistory`, чтобы агент запоминал диалог.

Теперь вы владеете **двумя подходами**: можете собрать RAG «с нуля» (как в Лекциях 6.1–6.2) и можете использовать мощь фреймворка для быстрой разработки. Это даёт вам **глубокое понимание** и **инструментальную гибкость** — редкое сочетание!

### Что дальше?

В **Лекции 6.4** мы перейдём к созданию **настоящих агентов** — систем, которые не просто отвечают на вопросы, но и **принимают решения, используют инструменты** (калькулятор, поиск в интернете, вызов API) и **управляют сложными потоками**. Мы освоим **LangGraph** — фреймворк для построения графовых агентов с состоянием, ветвлениями и циклами. Это следующий уровень, где ваш RAG-агент превратится в полноценного цифрового помощника.


## Домашнее задание к лекции 6.3

В этой лекции мы перешли от ручного управления всеми компонентами RAG к использованию мощного фреймворка LangChain. Ваше домашнее задание — закрепить этот переход на практике: переписать свою систему с LangChain, поэкспериментировать с компонентами и сравнить результаты.

> **Важно:** все задания выполняются с использованием **вашей собственной базы Chroma**, созданной в Лекции 6.2. Если вы не делали Лекцию 6.2, создайте хотя бы минимальный набор документов (3–5 файлов) и проиндексируйте их с помощью эмбеддингов `all-MiniLM-L6-v2`.

---

### Обязательная часть (5 баллов)

#### 1. Миграция на LangChain (3 балла)

Возьмите ваш код из Лекции 6.2 (полноценный агент с маршрутизацией, памятью и RAG) и **перепишите его с использованием LangChain** по образцу из раздела «Итоговый код: компактный RAG-агент» (конец лекции). Ваша новая реализация должна:

- Загружать векторную базу через `Chroma` из `langchain_chroma` с правильным именем коллекции.
- Использовать `HuggingFaceEmbeddings` (не `SentenceTransformerEmbeddings`).
- Собирать RAG-цепочку через LCEL (`RunnablePassthrough.assign`, `|` оператор).
- Добавить память с помощью `InMemoryChatMessageHistory` и `RunnableWithMessageHistory`.
- Использовать `StrOutputParser` для извлечения ответа.

**Что сдать:**
- Файл `rag_langchain.py` с рабочим кодом.
- Скриншоты (или вывод терминала) работы агента на **трёх разных вопросах** (один простой по документам, один общий, один уточняющий, требующий памяти).

**Дополнительный вопрос (в отчёте):**  
Сравните объём кода (количество строк) вашей новой реализации с кодом из Лекции 6.2. Укажите, насколько процентов сократился объём, и прокомментируйте, что стало проще / сложнее.

---

#### 2. Добавление постобработки ответа (2 балла)

Иногда LLM возвращает пустой ответ, или ответ, содержащий только "я не знаю", или чрезмерно длинный. Добавьте в вашу RAG-цепочку шаг **постобработки**:

- После получения ответа от LLM (после парсера) проверьте:
  - Если ответ пустой (`len(answer.strip()) == 0`) или содержит фразу `"не знаю"` / `"не могу"` (регистронезависимо), замените его на заранее заготовленное сообщение: `"Извините, я не нашёл информации по вашему вопросу в документах."`
- Реализуйте это через **дополнительный раннабл** в цепочке (например, `RunnableLambda`), который принимает строку и возвращает строку.

**Что сдать:**
- Код с постобработкой.
- Демонстрацию работы на вопросе, на который в документах нет ответа (например, "Как зовут директора компании?").

**Подсказка:** можно использовать `RunnableLambda`:

```python
from langchain_core.runnables import RunnableLambda

def postprocess(answer: str) -> str:
    if not answer.strip() or "не знаю" in answer.lower():
        return "Извините, я не нашёл информации по вашему вопросу в документах."
    return answer

rag_chain = (...) | StrOutputParser() | RunnableLambda(postprocess)
```

---

### Дополнительная часть (эксперименты — выполните минимум 3 из 5, каждый до 2 баллов)

#### 3. Сравнение моделей эмбеддингов (2 балла)

В лекции мы использовали `all-MiniLM-L6-v2`. Попробуйте заменить её на другую модель, например:

- `paraphrase-multilingual-MiniLM-L12-v2` — лёгкая мультиязычная.
- `intfloat/multilingual-e5-small` — современная, компактная.
- `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` — качественная, но более тяжёлая.

**Что сделать:**
- Создайте две отдельные векторные базы (или пересоздайте одну) с разными эмбеддингами.
- Для **одного и того же вопроса** сравните топ‑3 чанка, найденные каждой моделью (выведите их тексты и оценки релевантности).
- Замерьте время загрузки модели и инференса (используйте `time.time()`).
- В отчёте напишите, какая модель дала лучший результат по смыслу на ваших документах, и объясните, почему.

---

#### 4. Улучшенный маршрутизатор с PydanticOutputParser (2 балла)

В лекции мы реализовали маршрутизатор на основе `PydanticOutputParser`. Ваша задача — **доработать его**:

- Добавьте в структуру `RouterOutput` ещё одно поле: `reason` (строка с кратким объяснением, почему выбрано именно это действие).
- Измените промпт маршрутизатора так, чтобы он выдавал не только `action` и `confidence`, но и `reason`.
- Протестируйте на **10 разных вопросах** (5 требующих поиска, 5 не требующих) и зафиксируйте процент правильных решений.
- Сравните точность с маршрутизацией по ключевым словам (из Лекции 6.1) — насколько улучшилась?

**Что сдать:** код, примеры вопросов и ответов, статистику точности.

---

#### 5. Обрезка истории (оконное окно) (2 балла)

В долгих диалогах история может стать слишком длинной. Реализуйте **обрезку истории** до последних N пар сообщений (например, хранить только последние 3 вопроса и ответа).

- Модифицируйте функцию `get_session_history` так, чтобы она возвращала историю, обрезанную до заданного числа пар.
- Сравните качество ответов на уточняющие вопросы при `N=1`, `N=3`, `N=5`. Оцените, при каком N ответы становятся неудовлетворительными, а при каком — оптимальными.
- Сделайте вывод о необходимом размере окна для вашей предметной области.

---

#### 6. Логирование всех шагов цепочки (2 балла)

Добавьте в вашу RAG-цепочку **логирование** каждого ключевого шага:

- Входной вопрос.
- Найденные релевантные чанки (их содержимое и оценки, если возможно).
- Сформированный промпт (полный текст, отправленный в LLM).
- Ответ модели.

Используйте стандартный модуль `logging` с уровнем `DEBUG`. Настройте вывод в файл и в консоль.

**Что сдать:** код с логированием и пример лог-файла для одного диалога (3–4 вопроса). Объясните, как это помогает отладке.

---

#### 7. Замена LLM на другую модель (2 балла)

В лекции мы использовали `qwen2.5:3b`. Установите через `ollama pull` другую модель (например, `llama3.1:8b`, `mistral:7b`, `deepseek-r1:7b`) и замените её в вашей цепочке.

**Что сравнить:**
- Качество ответа (точность следования инструкции, полноту, отсутствие галлюцинаций).
- Скорость генерации.
- Потребление памяти (можно оценить по `nvidia-smi` или `htop`).

Сделайте **три одинаковых вопроса** для каждой модели и запишите ответы. В отчёте напишите, какая модель показала себя лучше в вашем сценарии и почему.

---

### Формат сдачи

- Код каждого выполненного пункта сохраните в отдельном файле (или в одном файле с комментариями).
- Подготовьте **отчёт** в формате `.md` или `.pdf`, в котором:
  - Для каждого пункта опишите, что сделано, приведите код (если требуется), скриншоты или текстовые выводы.
  - Для пунктов с экспериментами обязательно включите таблицы сравнения и ваши выводы.
- Загрузите всё в один архив или используйте Git-репозиторий.

---

### Критерии оценки

| Раздел | Максимум баллов |
|--------|----------------|
| **Обязательная часть** | 5 |
| Миграция на LangChain (код + сравнение) | 3 |
| Постобработка ответа (код + демонстрация) | 2 |
| **Дополнительная часть** (каждый пункт) | до 2 (максимум +10) |
| За каждый выполненный пункт: 1 балл за реализацию, 1 балл за анализ/выводы | |
| **Итого** | 15 |

**Штрафы:**
- Отсутствие отчёта или невнятные выводы — минус 2 балла.
- Код без комментариев и с плохим форматированием — минус 1 балл.

---

### Рекомендации

- Начинайте с обязательной части — она даёт базовое понимание.
- Для экспериментов используйте те же вопросы, чтобы результаты были сопоставимы.
- Не бойтесь модифицировать код и пробовать разные параметры — это и есть главная цель ДЗ.
- Если возникают ошибки (например, `DeprecationWarning`), постарайтесь разобраться и использовать актуальные импорты.
